# 04b — Continuous Outcome Models (Ridge Regression)

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/continuous_ridge_summary.csv`, `../outputs/tables/continuous_ridge_perfold.csv`

**Description:**
- Ridge regression predicting continuous crisis/DSI scores (not binary cutoffs)
- Concurrent: PM text → same-day PM crisis total, PM text → DSI-SS total
- Baseline (numeric only) vs Full (numeric + text PCA)
- GroupKFold CV with per-fold MAE, RMSE, R² and standard deviations
- Matches original cells 6-10, 13, 38

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
CRISIS_COL = "crisis_PM_from_full"

N_SPLITS = 5
N_PCS = 20
RIDGE_ALPHA = 10.0
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [3]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day), "Row mismatch"
print("Loaded data:", pm_day.shape)
print("Loaded embeddings:", X_text.shape)

groups = pm_day[PID_COL].astype(str).values

Loaded data: (2511, 73)
Loaded embeddings: (2511, 768)


In [4]:
# =========================
# BUILD NUMERIC BASELINE
# =========================
num_candidates = [
    c for c in pm_day.columns
    if c.startswith("dailyPM__") and pd.api.types.is_numeric_dtype(pm_day[c])
]
baseline_cols = [CRISIS_COL] + num_candidates

X_num_df = pm_day[baseline_cols].copy()
X_num_df = X_num_df.apply(pd.to_numeric, errors="coerce")
X_num_df = X_num_df.dropna(axis=1, how="all")
X_num = X_num_df.fillna(X_num_df.mean()).values.astype(float)

# Safety: drop columns still containing NaN
nan_cols = np.any(np.isnan(X_num), axis=0)
if nan_cols.any():
    print(f"Dropping {nan_cols.sum()} columns still containing NaN")
    X_num = X_num[:, ~nan_cols]

print("Baseline features:", X_num.shape[1])

Baseline features: 15


In [8]:
# =========================
# CV HELPERS
# =========================
def grouped_cv_ridge(X, y, groups, n_splits=5, label=""):
    gkf = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
    maes, rmses, r2s = [], [], []
    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED))
        ])
        pipe.fit(X[tr], y[tr])
        yhat = pipe.predict(X[te])
        maes.append(mean_absolute_error(y[te], yhat))
        rmses.append(np.sqrt(mean_squared_error(y[te], yhat)))
        r2s.append(r2_score(y[te], yhat))
        print(f"  {label} fold {fold}: MAE={maes[-1]:.3f} RMSE={rmses[-1]:.3f} R²={r2s[-1]:.3f}")
    return {
        "MAE_mean": float(np.mean(maes)), "MAE_sd": float(np.std(maes)),
        "RMSE_mean": float(np.mean(rmses)), "RMSE_sd": float(np.std(rmses)),
        "R2_mean": float(np.mean(r2s)), "R2_sd": float(np.std(r2s)),
    }


def grouped_cv_text_plus_numeric(X_text, X_num, y, groups, n_splits=5, n_pcs=20, label=""):
    gkf = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
    maes, rmses, r2s = [], [], []
    for fold, (tr, te) in enumerate(gkf.split(X_text, y, groups=groups), start=1):
        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])

        Xtr = np.hstack([X_num[tr], Ttr])
        Xte = np.hstack([X_num[te], Tte])

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_SEED))
        ])
        pipe.fit(Xtr, y[tr])
        yhat = pipe.predict(Xte)
        maes.append(mean_absolute_error(y[te], yhat))
        rmses.append(np.sqrt(mean_squared_error(y[te], yhat)))
        r2s.append(r2_score(y[te], yhat))
        print(f"  {label} fold {fold}: MAE={maes[-1]:.3f} RMSE={rmses[-1]:.3f} R²={r2s[-1]:.3f}")
    return {
        "MAE_mean": float(np.mean(maes)), "MAE_sd": float(np.std(maes)),
        "RMSE_mean": float(np.mean(rmses)), "RMSE_sd": float(np.std(rmses)),
        "R2_mean": float(np.mean(r2s)), "R2_sd": float(np.std(r2s)),
    }

In [10]:
# =========================
# RUN CONTINUOUS MODELS
# =========================
CONTINUOUS_OUTCOMES = {
    "crisis_PM_total": CRISIS_COL,
    "dsi_PM_total": "dsi_PM_total",
}

all_rows = []

for outcome_label, outcome_col in CONTINUOUS_OUTCOMES.items():
    if outcome_col not in pm_day.columns:
        print(f"Skipping {outcome_label}: column {outcome_col} not found")
        continue

    y = pm_day[outcome_col].astype(float).values
    mask = np.isfinite(y) & np.isfinite(X_num).all(axis=1)
    y_m = y[mask]
    X_num_m = X_num[mask]
    X_text_m = X_text[mask]
    groups_m = groups[mask]

    print(f"\n{'='*60}")
    print(f"Outcome: {outcome_label} | N={len(y_m)} | mean={y_m.mean():.3f} | sd={y_m.std():.3f}")
    print(f"{'='*60}")

    # Baseline: numeric only
    print("\n--- Baseline (numeric only) ---")
    res_base = grouped_cv_ridge(X_num_m, y_m, groups_m, N_SPLITS, label="Baseline")

    # Full: numeric + text
    print("\n--- Full (numeric + text PCA) ---")
    res_full = grouped_cv_text_plus_numeric(
        X_text_m, X_num_m, y_m, groups_m, N_SPLITS, N_PCS, label="Full"
    )

    delta = {
        "Delta_MAE": res_base["MAE_mean"] - res_full["MAE_mean"],
        "Delta_RMSE": res_base["RMSE_mean"] - res_full["RMSE_mean"],
        "Delta_R2": res_full["R2_mean"] - res_base["R2_mean"],
    }

    print(f"\nIncremental gain: ΔMAE={delta['Delta_MAE']:.4f}, ΔRMSE={delta['Delta_RMSE']:.4f}, ΔR²={delta['Delta_R2']:.4f}")

    all_rows.append({"outcome": outcome_label, "model": "baseline", "n": len(y_m), **res_base})
    all_rows.append({"outcome": outcome_label, "model": "full_text", "n": len(y_m), **res_full})
    all_rows.append({"outcome": outcome_label, "model": "delta", "n": len(y_m), **delta})


Outcome: crisis_PM_total | N=2511 | mean=6.149 | sd=6.233

--- Baseline (numeric only) ---
  Baseline fold 1: MAE=0.039 RMSE=0.054 R²=1.000
  Baseline fold 2: MAE=0.049 RMSE=0.070 R²=1.000
  Baseline fold 3: MAE=0.041 RMSE=0.060 R²=1.000
  Baseline fold 4: MAE=0.045 RMSE=0.067 R²=1.000
  Baseline fold 5: MAE=0.038 RMSE=0.055 R²=1.000

--- Full (numeric + text PCA) ---
  Full fold 1: MAE=0.046 RMSE=0.061 R²=1.000
  Full fold 2: MAE=0.054 RMSE=0.078 R²=1.000
  Full fold 3: MAE=0.046 RMSE=0.065 R²=1.000
  Full fold 4: MAE=0.051 RMSE=0.072 R²=1.000
  Full fold 5: MAE=0.039 RMSE=0.056 R²=1.000

Incremental gain: ΔMAE=-0.0048, ΔRMSE=-0.0051, ΔR²=-0.0000

Outcome: dsi_PM_total | N=2511 | mean=1.724 | sd=2.581

--- Baseline (numeric only) ---
  Baseline fold 1: MAE=1.345 RMSE=1.977 R²=0.439
  Baseline fold 2: MAE=1.616 RMSE=2.254 R²=0.392
  Baseline fold 3: MAE=1.267 RMSE=1.797 R²=0.354
  Baseline fold 4: MAE=1.435 RMSE=1.843 R²=0.352
  Baseline fold 5: MAE=1.481 RMSE=1.964 R²=0.407

--- Full

In [12]:
# =========================
# SAVE
# =========================
summary = pd.DataFrame(all_rows)
summary_path = os.path.join(OUT_DIR, "continuous_ridge_summary.csv")
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)
print(summary.to_string(index=False))


Saved: ..\outputs\tables\continuous_ridge_summary.csv
        outcome     model    n  MAE_mean   MAE_sd  RMSE_mean  RMSE_sd  R2_mean    R2_sd  Delta_MAE  Delta_RMSE  Delta_R2
crisis_PM_total  baseline 2511  0.042416 0.004217   0.061097 0.006583 0.999899 0.000030        NaN         NaN       NaN
crisis_PM_total full_text 2511  0.047213 0.004984   0.066233 0.007648 0.999881 0.000035        NaN         NaN       NaN
crisis_PM_total     delta 2511       NaN      NaN        NaN      NaN      NaN      NaN  -0.004796   -0.005136 -0.000018
   dsi_PM_total  baseline 2511  1.428922 0.119093   1.967166 0.159310 0.388779 0.032682        NaN         NaN       NaN
   dsi_PM_total full_text 2511  1.478681 0.096868   2.007148 0.139909 0.355170 0.111587        NaN         NaN       NaN
   dsi_PM_total     delta 2511       NaN      NaN        NaN      NaN      NaN      NaN  -0.049759   -0.039982 -0.033608
